# QUELL 05.6 — Matched baseline (dengeli train + aynı eval, çok-seed)

RF/XGB'yi LLM ile **aynı** rejimde (dengeli 2000/sınıf train, aynı 60k stratified eval) CICIoT2023 + N-BaIoT'ta çok-seedle skorlar → üç veri seti tutarlı, adil kıyas. LLM yok, hızlı. Tek hücre; OZET'i paylaş.

In [ ]:
# ===== QUELL 05.6 — Baseline'lari LLM ile AYNI rejimde yeniden skorla (matched, cok-seed) =====
# Goal: also measure RF/XGB on a balanced 2000/class train and the SAME 60k eval as the LLM, for a fair comparison.
# CICIoT2023 + N-BaIoT (Edge already done in 05.5). No LLM -> fast.
import os, json, time
from pathlib import Path
import numpy as np, pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import xgboost as xgb

DATASETS=["ciciot2023","nbaiot"]
SEEDS=[42,1,2,3]; TRAIN_CAP=2000; EVAL_CAP=60000; EVAL_SEED=42   # eval: same setup as the 05 LLM
ROOT=Path.home()/"quell-edge-llm-ids"; PROC=ROOT/"data"/"processed"; SPL=ROOT/"splits"; RES=ROOT/"results"
rep=json.load(open(RES/"split_report.json"))
LABELISH={"label","attack","attack_type","attack_label","type","class","category","marker","__label__"}
OUT=RES/"baseline_matched_report.json"
allr=json.load(open(OUT)) if OUT.exists() else {}

for DATASET in DATASETS:
    meta=rep[DATASET]; label=meta["label_col"]; group=meta.get("group_col"); tcol=meta.get("time_col")
    df=pd.read_parquet(PROC/f"{DATASET}.parquet").reset_index(drop=True)
    sp=np.load(SPL/f"{DATASET}_split.npz"); tr_idx,te_idx=sp["train"],sp["test"]
    drop=set([label])|set(meta.get("leaky_candidates",[]))
    if group: drop.add(group)
    if tcol: drop.add(tcol)
    for c in df.columns:
        if c!=label and c.lower() in LABELISH: drop.add(c)
    feats=[c for c in df.columns if c not in drop]
    y=df[label].astype(str).values; classes=sorted(pd.unique(y).tolist())
    Xdf=df[feats].copy()
    for c in Xdf.columns:
        if not is_numeric_dtype(Xdf[c]): Xdf[c]=pd.factorize(Xdf[c])[0]
    X=np.nan_to_num(Xdf.values.astype("float32"))
    # LLM ile AYNI eval (60k stratified, sabit EVAL_SEED)
    if len(te_idx)<=EVAL_CAP: eval_idx=te_idx
    else:
        r=np.random.default_rng(EVAL_SEED); pick=[]
        for cls in np.unique(y[te_idx]):
            ids=te_idx[y[te_idx]==cls]; k=max(1,int(round(len(ids)*EVAL_CAP/len(te_idx))))
            pick+=r.choice(ids,min(k,len(ids)),replace=False).tolist()
        eval_idx=np.array(sorted(pick))
    yte=y[eval_idx]
    print(f"\n=== {DATASET} | feature={len(feats)} class={len(classes)} eval={len(eval_idx):,} ===",flush=True)
    rf_s=[]; xgb_s=[]
    for seed in SEEDS:
        rng=np.random.default_rng(seed); sel=[]
        for cls in pd.unique(y[tr_idx]):
            ids=tr_idx[y[tr_idx]==cls]
            if len(ids)>TRAIN_CAP: ids=rng.choice(ids,TRAIN_CAP,replace=False)
            sel+=ids.tolist()
        sel=np.array(sorted(sel))
        rf=RandomForestClassifier(n_estimators=300,n_jobs=-1,class_weight="balanced",random_state=seed).fit(X[sel],y[sel])
        rf_f1=f1_score(yte,rf.predict(X[eval_idx]),average="macro",labels=classes,zero_division=0)
        le=LabelEncoder().fit(y[sel])
        xg=xgb.XGBClassifier(n_estimators=300,max_depth=8,n_jobs=-1,tree_method="hist",random_state=seed,
            num_class=len(le.classes_),objective="multi:softmax",eval_metric="mlogloss").fit(X[sel],le.transform(y[sel]))
        xg_f1=f1_score(yte,le.inverse_transform(xg.predict(X[eval_idx])),average="macro",labels=classes,zero_division=0)
        rf_s.append(float(rf_f1)); xgb_s.append(float(xg_f1))
        print(f"  seed {seed}: RF={rf_f1:.4f}  XGB={xg_f1:.4f}",flush=True)
    allr[DATASET]={"regime":"balanced 2000/class, matched 60k eval (EVAL_SEED=42)","seeds":SEEDS,
        "rf":{"mean":round(float(np.mean(rf_s)),4),"std":round(float(np.std(rf_s)),4),"seeds":[round(v,4) for v in rf_s]},
        "xgb":{"mean":round(float(np.mean(xgb_s)),4),"std":round(float(np.std(xgb_s)),4),"seeds":[round(v,4) for v in xgb_s]}}
    json.dump(allr,open(OUT,"w"),indent=2,ensure_ascii=False)
    print(f"  RF={allr[DATASET]['rf']['mean']}±{allr[DATASET]['rf']['std']}  XGB={allr[DATASET]['xgb']['mean']}±{allr[DATASET]['xgb']['std']}",flush=True)

print("\n===== MATCHED BASELINE OZET (macro-F1, ort±std) =====")
for d in DATASETS:
    print(f"  {d:12s} RF={allr[d]['rf']['mean']}±{allr[d]['rf']['std']}  XGB={allr[d]['xgb']['mean']}±{allr[d]['xgb']['std']}")
print("saved -> results/baseline_matched_report.json | DONE")
